# Backtracking — Cheat Sheet

**Companion to:** DP Study Guide (contrast between Backtracking and DP is covered in Part 0 of that guide)

**What this adds:** Backtracking was not covered in any study guide. This cheat sheet gives you the universal template, the four problem types, pruning strategies, and the decision guide.

---
## When to Use

| Signal | Pattern to reach for |
| :--- | :--- |
| "all permutations", "all arrangements" | Backtracking — Permutations |
| "all subsets", "power set" | Backtracking — Subsets |
| "all combinations that sum to target" | Backtracking — Combinations |
| "place N queens", "valid board configurations" | Backtracking — Constraint satisfaction |
| "partition into palindromes", "word break all ways" | Backtracking — String partition |
| "find ALL solutions" (not just one or the count) | Backtracking over DP |
| Choices at each step with no overlapping subproblems | Backtracking over DP |

---
## Patterns at a Glance

| Pattern | Start index needed? | Can reuse elements? | Key pruning | Problems |
| :--- | :--- | :--- | :--- | :--- |
| Permutations | No — use `visited` set | No | Skip visited indices | 46, 47 |
| Subsets | Yes — `start` index | No | None needed | 78, 90 |
| Combinations | Yes — `start` index | Depends | Skip duplicates; bound check | 39, 40, 77 |
| Constraint satisfaction | Varies | No | Validity check before recursing | 51, 37 |
| String partition | Yes — position index | No | Validity check on slice | 131, 93 |

---
# Part 0 — Quick Recap

Backtracking is a recursive strategy that **builds a solution incrementally**, abandoning a path the moment it violates a constraint (pruning), and undoing the last choice to try the next option (backtracking).

**How it differs from DP:** both use recursion, but DP caches results because subproblems overlap. In backtracking every path through the search space is unique — there is nothing to cache. If you find yourself recomputing the same subproblem, switch to DP.

**How it differs from DFS on a graph:** graph DFS visits existing nodes and edges. Backtracking builds the search tree on the fly — each recursive call represents a choice, and the "graph" only exists implicitly.

**The universal template has six steps — always write them as comments before coding:**
```
1. Base case      → when to record the answer and return
2. Loop           → iterate over all valid choices at this step
3. Prune          → skip invalid choices before recursing
4. Choose         → make the choice (add to path, mark visited)
5. Recurse        → move one step deeper
6. Unchoose       → undo the choice (backtrack)
```

---
# Part 1 — Universal Template

## Core idea

Every backtracking problem uses the same six-step skeleton. The only things that change between problem types are: what the base case checks, what the loop iterates over, and what the pruning condition is.

In [ ]:
def backtrack(path, choices, res):
    # 1. Base case — record answer when path is complete
    if BASE_CASE:
        res.append(path[:])     # append a COPY — path is mutated in place
        return

    for choice in choices:
        # 2. Prune — skip invalid choices BEFORE recursing
        if NOT_VALID(choice):
            continue

        # 3. Choose
        path.append(choice)

        # 4. Recurse — move one step deeper
        backtrack(path, NEXT_CHOICES, res)

        # 5. Unchoose — undo the choice (backtrack)
        path.pop()

res = []
backtrack([], ALL_CHOICES, res)
return res

## Common mistakes

- Appending `path` instead of `path[:]` — all results end up as the same empty list since `path` is mutated throughout
- Forgetting to unchoose — the function works correctly on the first branch but corrupts subsequent branches
- Pruning after recursing instead of before — wasted recursive calls; always check validity before going deeper

---
# Part 2 — Permutations

## Core idea

Order matters — `[1, 2]` and `[2, 1]` are different. Every element is used exactly once. Track which elements have been used with a `visited` set or boolean array instead of a `start` index — a `start` index only works when order doesn't matter.

**Duplicate handling (LC 47):** sort the input first. Skip a choice if it equals the previous choice AND the previous choice has not been used in the current path — this prevents generating the same permutation twice from two identical values.

In [ ]:
# Template — Permutations (no duplicates, LC 46)
def permute(nums):
    res, path, visited = [], [], set()

    def backtrack():
        if len(path) == len(nums):  # base case: path is full
            res.append(path[:])
            return
        for i, num in enumerate(nums):
            if i in visited: continue   # skip already-used elements
            visited.add(i)
            path.append(num)
            backtrack()
            path.pop()
            visited.remove(i)

    backtrack()
    return res


# Template — Permutations with duplicates (LC 47)
def permuteUnique(nums):
    nums.sort()                         # sort to group duplicates together
    res, path, visited = [], [], [False] * len(nums)

    def backtrack():
        if len(path) == len(nums):
            res.append(path[:])
            return
        for i in range(len(nums)):
            if visited[i]: continue
            # skip duplicate: same value as previous AND previous not used in this path
            if i > 0 and nums[i] == nums[i-1] and not visited[i-1]: continue
            visited[i] = True
            path.append(nums[i])
            backtrack()
            path.pop()
            visited[i] = False

    backtrack()
    return res

## Common mistakes

- Using a `start` index for permutations — a start index prevents reusing earlier elements, but permutations need all elements at every position; use a `visited` set instead
- Forgetting to `visited.remove(i)` on backtrack — subsequent branches incorrectly treat that index as used
- In LC 47: sorting but skipping `if not visited[i-1]` — this skips too aggressively and misses valid permutations

**Practice problems:**

| Problem | Notes |
| :--- | :--- |
| LC 46 — Permutations | No duplicates — use visited set |
| LC 47 — Permutations II | Duplicates — sort + skip condition |
| LC 60 — Permutation Sequence | Find kth permutation — math shortcut more efficient than full backtrack |

---
# Part 3 — Subsets & Combinations

## Core idea

Order does NOT matter — `[1, 2]` and `[2, 1]` are the same. Use a `start` index to ensure elements are only picked in forward order, preventing duplicates.

**Subsets vs Combinations:**
- **Subsets** — record the path at every node (not just the leaves); no target sum
- **Combinations** — record only when path meets a condition (length = k, or sum = target)

**Reuse rule:**
- Elements used **at most once** → next call starts at `start + 1`
- Elements reusable **unlimited times** → next call starts at `start` (same index)

In [ ]:
# Template — Subsets (LC 78, no duplicates)
def subsets(nums):
    res, path = [], []

    def backtrack(start):
        res.append(path[:])             # record at EVERY node, not just leaves
        for i in range(start, len(nums)):
            path.append(nums[i])
            backtrack(i + 1)            # i+1: each element used at most once
            path.pop()

    backtrack(0)
    return res


# Template — Subsets with duplicates (LC 90)
def subsetsWithDup(nums):
    nums.sort()                         # sort to group duplicates
    res, path = [], []

    def backtrack(start):
        res.append(path[:])
        for i in range(start, len(nums)):
            if i > start and nums[i] == nums[i-1]: continue  # skip duplicate at same level
            path.append(nums[i])
            backtrack(i + 1)
            path.pop()

    backtrack(0)
    return res


# Template — Combination Sum, elements reusable (LC 39)
def combinationSum(candidates, target):
    res, path = [], []

    def backtrack(start, remaining):
        if remaining == 0:              # base case: exact match
            res.append(path[:])
            return
        for i in range(start, len(candidates)):
            if candidates[i] > remaining: break  # pruning: sorted array, no point continuing
            path.append(candidates[i])
            backtrack(i, remaining - candidates[i])  # i not i+1: same element reusable
            path.pop()

    candidates.sort()                   # sort enables pruning
    backtrack(0, target)
    return res


# Template — Combination Sum, each element used once (LC 40)
def combinationSum2(candidates, target):
    candidates.sort()
    res, path = [], []

    def backtrack(start, remaining):
        if remaining == 0:
            res.append(path[:])
            return
        for i in range(start, len(candidates)):
            if candidates[i] > remaining: break
            if i > start and candidates[i] == candidates[i-1]: continue  # skip duplicate at same level
            path.append(candidates[i])
            backtrack(i + 1, remaining - candidates[i])  # i+1: each element used once
            path.pop()

    backtrack(0, target)
    return res

## Common mistakes

- Confusing `i` vs `i + 1` in the recursive call — `i` allows reuse, `i + 1` prevents it; mixing these up generates wrong results silently
- Duplicate skipping condition `i > start` vs `i > 0` — use `i > start` (not `i > 0`) to only skip duplicates at the same recursion level, not across levels
- Forgetting to sort before pruning with `if candidates[i] > remaining: break` — the break only works correctly on a sorted array

**Practice problems:**

| Problem | Record at | Reuse | Notes |
| :--- | :--- | :--- | :--- |
| LC 78 — Subsets | Every node | No | No target condition |
| LC 90 — Subsets II | Every node | No | Sort + skip duplicates at same level |
| LC 39 — Combination Sum | Leaves only (sum == target) | Yes | Next call at `i`, not `i+1` |
| LC 40 — Combination Sum II | Leaves only | No | Sort + skip; next call at `i+1` |
| LC 77 — Combinations | Leaves only (len == k) | No | No target sum — stop at length k |

---
# Part 4 — Constraint Satisfaction

## Core idea

Place elements one position at a time. Before placing, check whether the placement is valid given what's already on the board. The pruning is the validity check — if placing here violates a constraint, skip it entirely.

**Key insight:** encode the constraints as sets for O(1) lookup. For N-Queens, track which columns, diagonal directions (`row - col`), and anti-diagonal directions (`row + col`) are already occupied.

In [ ]:
# Template — N-Queens (LC 51)
def solveNQueens(n):
    res = []
    cols    = set()             # occupied columns
    diag1   = set()             # occupied diagonals (row - col is constant on a diagonal)
    diag2   = set()             # occupied anti-diagonals (row + col is constant)

    board = ["." * n for _ in range(n)]

    def backtrack(row):
        if row == n:            # base case: all rows placed
            res.append(board[:])
            return
        for col in range(n):
            # prune: skip if column or either diagonal is occupied
            if col in cols or (row - col) in diag1 or (row + col) in diag2:
                continue
            # choose
            cols.add(col)
            diag1.add(row - col)
            diag2.add(row + col)
            board[row] = "." * col + "Q" + "." * (n - col - 1)
            # recurse: move to next row
            backtrack(row + 1)
            # unchoose
            cols.remove(col)
            diag1.remove(row - col)
            diag2.remove(row + col)
            board[row] = "." * n

    backtrack(0)
    return res

## Common mistakes

- Checking constraints after placing instead of before — wastes recursive calls; always prune before choosing
- Using a 2D board scan to check validity — O(n) per check; use sets for O(1)
- Forgetting to undo all three constraint sets on backtrack — stale entries cause valid placements to be incorrectly skipped

**Practice problems:**

| Problem | Constraint encoded as | Notes |
| :--- | :--- | :--- |
| LC 51 — N-Queens | Three sets: col, diag, anti-diag | One queen per row — iterate columns only |
| LC 52 — N-Queens II | Same three sets | Count only, not the board configurations |
| LC 37 — Sudoku Solver | Sets per row, col, 3×3 box | Place digit; if no valid digit exists, return False |

---
# Part 5 — String Partition

## Core idea

Partition a string by iterating over all possible split points. At each position, check if the slice from `start` to `end` is valid. If yes, add it to the path and recurse from `end`. Backtrack when done.

**Key insight:** the loop variable `end` acts as both the split point and the next `start` index for the recursive call — `backtrack(end)` not `backtrack(end + 1)`.

In [ ]:
# Template — String Partition
def partition_string(s, is_valid):
    res, path = [], []

    def backtrack(start):
        if start == len(s):         # base case: consumed entire string
            res.append(path[:])
            return
        for end in range(start + 1, len(s) + 1):   # end is exclusive
            segment = s[start:end]
            if not is_valid(segment): continue      # prune invalid segments
            path.append(segment)
            backtrack(end)                          # next start = end (not end+1)
            path.pop()

    backtrack(0)
    return res


# LC 131 — Palindrome Partitioning
def partition(s):
    res, path = [], []

    def is_palindrome(sub):
        return sub == sub[::-1]     # O(n) check — precompute with DP for large inputs

    def backtrack(start):
        if start == len(s):
            res.append(path[:])
            return
        for end in range(start + 1, len(s) + 1):
            if not is_palindrome(s[start:end]): continue
            path.append(s[start:end])
            backtrack(end)
            path.pop()

    backtrack(0)
    return res

## Common mistakes

- Using `backtrack(end + 1)` instead of `backtrack(end)` — skips one character between segments, producing wrong partitions
- Checking `s[start:end]` with `end` inclusive — Python slicing is exclusive on the right; `s[start:end]` already excludes index `end`
- Recomputing palindrome check inside backtrack on every call — precompute a 2D DP table `is_pal[i][j]` if the input is large

**Practice problems:**

| Problem | Valid segment condition | Notes |
| :--- | :--- | :--- |
| LC 131 — Palindrome Partitioning | `s[start:end]` is a palindrome | Precompute palindrome table for optimization |
| LC 93 — Restore IP Addresses | Segment is valid IPv4 octet (0–255, no leading zeros) | Fixed depth: stop after 4 segments |

---
# Decision Guide

```
Does order matter in the output?
├── Yes → Permutations
│         ├── No duplicate inputs    → visited set, no start index    (LC 46)
│         └── Duplicate inputs       → sort + skip if prev not visited (LC 47)
│
└── No  → Order doesn't matter
          │
          ├── No target condition (collect everything)
          │   └── Subsets — record at every node, not just leaves
          │         ├── No duplicates   → start index, i+1            (LC 78)
          │         └── Duplicates      → sort + skip at same level   (LC 90)
          │
          ├── Target condition (sum, length)
          │   └── Combinations — record only at leaves
          │         ├── Elements reusable    → recurse at i           (LC 39)
          │         └── Elements used once   → sort + skip + i+1      (LC 40)
          │
          ├── Placing elements on a board with constraints
          │   └── Constraint satisfaction
          │         → encode constraints as sets, prune before placing (LC 51, 37)
          │
          └── Splitting a string into valid segments
              └── String partition
                    → loop end index, check segment validity, recurse(end) (LC 131, 93)

Should I use Backtracking or DP?
├── Need ALL solutions                        → Backtracking
├── Need count or optimal value only          → DP
├── Subproblems overlap (same state reached   → DP
│   via multiple paths)
└── Every path through the search space is   → Backtracking
    unique (no overlapping subproblems)
```